In [1]:

import numpy as np
import matplotlib.pyplot as plt
        

System's parameters

In [2]:

Omega_l=0.0031831 # configurational cost of forming loops q_l
Omega_b=9.2804*np.power(10.,-5) # ............................... bridges

z=4.
NDNA=75. 
emDG0=np.exp(20.) 
emDGT=np.exp(-3.) 

# surface's parameters
Omega_bs=0.000135192 # .............................. plane bridges

# BCC
z_inter=2.
z_intra=0
if(z != 2*z_inter+z_intra): 
    print('z_inter/intra not consistent with z')
    
Arec=44*38.105/16 
Nrec=1000./16 

## FCC
#z_inter=3
#z_intra=6
#Arec=44*38.105/16
#Nrec=100000./16 

equilib=False # notice that 'equilib' affect the bulk transition given that it affect the number of allowed 3-strands complexes


In [3]:

#
# return loops and bridges formed by a specific DNA strand on one particle
# x1:concentration of strand 1. In bulk x1=x2
#
def calculate_complexes_formed_by_1(x1,Omega_b,Omega_l,emDG0,emDGT):
    two_strands_bridges=x1*x1*Omega_b*emDG0
    three_strands_bridges=3*x1*x1*x1*Omega_b*Omega_l*emDG0*emDGT
    loops=x1*x1*Omega_l*emDG0
    return two_strands_bridges, three_strands_bridges, loops

#
# calculate the free energy per particle
#
def free_energy(x1,NDNA,Omega_b,Omega_l,emDG0,emDGT,z):
    two_strands_bridges, three_strands_bridges, loops = calculate_complexes_formed_by_1(x1,Omega_b,Omega_l,emDG0,emDGT)
    print('check convergence',NDNA,x1+z*two_strands_bridges+z*three_strands_bridges+loops)
    free_en=2.*NDNA*np.log(x1/NDNA)
    # loop contribution 
    free_en=free_en+loops
    # 2-bridges contribution (a factor 2 and a factor 1/2 compensate)
    free_en=free_en+two_strands_bridges*z
    # 3-bridges contribution (4/3: conunts the number of bridges, 2 -multimeric- and 1/2 compensate)
    free_en=free_en+three_strands_bridges*4./3.*z
    return free_en

#
# calculate the transition density
#
def transition_dens(x,x_ref,NDNA,Omega_b,Omega_l,emDG0,emDGT,z):
    cut=0.000000001
    rel_error=1.
    n_iter=0
    while(rel_error > cut):
        n_iter+=1
        x_old=x
        two_strands_bridges, three_strands_bridges, loops = calculate_complexes_formed_by_1(x,Omega_b,Omega_l,emDG0,emDGT)
        x=NDNA/(1+z*two_strands_bridges/x+z*three_strands_bridges/x+loops/x)
        rel_error=2.*np.abs(x-x_old)/(np.abs(x)+np.abs(x_old))
        if(np.mod(n_iter,np.power(10,6))==0):
            print(n_iter,rel_error, x, two_strands_bridges, three_strands_bridges, loops)

    rel_error=1.
    n_iter=0
    while(rel_error > cut):
        n_iter+=1
        x_ref_old=x_ref
        two_strands_bridges_ref, three_strands_bridges_ref, loops_ref = calculate_complexes_formed_by_1(x_ref,0.,Omega_l,emDG0,emDGT)
        x_ref=NDNA/(1+loops_ref/x_ref)
        rel_error=2.*np.abs(x_ref-x_ref_old)/(np.abs(x_ref)+np.abs(x_ref_old))
        if(np.mod(n_iter,np.power(10,6))==0):
            print(n_iter,rel_error, x_ref, two_strands_bridges_ref, three_strands_bridges_ref, loops_ref)

    f_e=free_energy(x,NDNA,Omega_b,Omega_l,emDG0,emDGT,z)
    f_e_ref=free_energy(x_ref,NDNA,0,Omega_l,emDG0,emDGT,0.)
    return x, x_ref, f_e, f_e_ref, np.exp(f_e-f_e_ref)



In [ ]:


x=0.0001
x_ref=x
print(transition_dens(x,x_ref,NDNA,Omega_b,Omega_l,emDG0,1./emDGT,z))


1000000 1.9999998159088956 1.154252336353792e-06 28321235.364982422 136237010.808294 971394813.6963441
2000000 1.9999998159088956 1.154252336353792e-06 28321235.364982422 136237010.808294 971394813.6963441
3000000 1.9999998159088956 1.154252336353792e-06 28321235.364982422 136237010.808294 971394813.6963441
4000000 1.9999998159088956 1.154252336353792e-06 28321235.364982422 136237010.808294 971394813.6963441
5000000 1.9999998159088956 1.154252336353792e-06 28321235.364982422 136237010.808294 971394813.6963441
6000000 1.9999998159088956 1.154252336353792e-06 28321235.364982422 136237010.808294 971394813.6963441
7000000 1.9999998159088956 1.154252336353792e-06 28321235.364982422 136237010.808294 971394813.6963441


Surface Theory 

In [ ]:

# returns (number of complexes with ligand a)/xa[i], a=1,2

def bottom_bridges(equilib,i_layer,x1,x2,Omega_b,Omega_bs,Omega_l,emDG0,emDGT,z_inter):

    if(i_layer == 0):
        nb1_bottom = 0.
        nb2_bottom = 0.
        n3b1_bottom = 0.
        n3b2_bottom = 0.
        n3tot = 0.
    else:
        if(i_layer == 1): 
            nb1_bottom = x1[i_layer-1]*Omega_bs*emDG0
            nb2_bottom = 0.
            n3b1_bottom = x2[i_layer]*x1[i_layer-1]*Omega_bs*Omega_l*emDG0*emDGT
            n3b2_bottom = x1[i_layer]*x1[i_layer-1]*Omega_bs*Omega_l*emDG0*emDGT
            n3tot = x1[i_layer]*n3b1_bottom
        else:
            if(equilib==True):
                nb1_bottom = z_inter*x1[i_layer-1]*Omega_b*emDG0
                nb2_bottom = z_inter*x2[i_layer-1]*Omega_b*emDG0
                n3b1_bottom = z_inter*(x2[i_layer]*(x1[i_layer-1]+x2[i_layer-1])+x1[i_layer-1]*x2[i_layer-1])*Omega_b*Omega_l*emDG0*emDGT
                n3b2_bottom = z_inter*(x1[i_layer]*(x1[i_layer-1]+x2[i_layer-1])+x1[i_layer-1]*x2[i_layer-1])*Omega_b*Omega_l*emDG0*emDGT
                n3tot = x1[i_layer]*x2[i_layer]*(x1[i_layer-1]+x2[i_layer-1])+x1[i_layer-1]*x2[i_layer-1]*(x1[i_layer]+x2[i_layer])
                n3tot *= z_inter*Omega_b*Omega_l*emDG0*emDGT
            else:
                if(np.mod(i_layer,2)==0):
                    nb1_bottom = z_inter*x1[i_layer-1]*Omega_b*emDG0
                    nb2_bottom = z_inter*x2[i_layer-1]*Omega_b*emDG0
                    n3b1_bottom = z_inter*(x2[i_layer]*x2[i_layer-1]+x1[i_layer-1]*x2[i_layer-1])*Omega_b*Omega_l*emDG0*emDGT
                    n3b2_bottom = z_inter*(x1[i_layer]*x2[i_layer-1])*Omega_b*Omega_l*emDG0*emDGT
                    n3tot = x1[i_layer]*x2[i_layer]*x2[i_layer-1]+x1[i_layer-1]*x2[i_layer-1]*x1[i_layer]
                    n3tot *= z_inter*Omega_b*Omega_l*emDG0*emDGT
                else:
                    nb1_bottom = z_inter*x1[i_layer-1]*Omega_b*emDG0
                    nb2_bottom = z_inter*x2[i_layer-1]*Omega_b*emDG0
                    n3b2_bottom = z_inter*(x1[i_layer]*x1[i_layer-1]+x1[i_layer-1]*x2[i_layer-1])*Omega_b*Omega_l*emDG0*emDGT
                    n3b1_bottom = z_inter*(x2[i_layer]*x1[i_layer-1])*Omega_b*Omega_l*emDG0*emDGT
                    n3tot = x1[i_layer]*x2[i_layer]*x1[i_layer-1]+x1[i_layer-1]*x2[i_layer-1]*x2[i_layer]
                    n3tot *= z_inter*Omega_b*Omega_l*emDG0*emDGT
                    
    return nb1_bottom, nb2_bottom, n3b1_bottom, n3b2_bottom, n3tot

def top_bridges(equilib,i_layer,x1,x2,Omega_b,Omega_bs,Omega_l,emDG0,emDGT,z_inter):

    if(i_layer == NL-1):
        nb1_top = 0.
        nb2_top = 0.
        n3b1_top = 0. 
        n3b2_top = 0.
        n3tot = 0.
    else:
        if(i_layer == 0):
            nb1_top = x1[i_layer+1]*Omega_bs*emDG0
            nb2_top = 0.
            n3b2_top = 0. 
            n3b1_top = x2[i_layer+1]*x1[i_layer+1]*Omega_bs*Omega_l*emDG0*emDGT
            n3tot = n3b1_top*x1[i_layer]
        else:
            if(equilib == True):
                nb1_top = z_inter*x1[i_layer+1]*Omega_b*emDG0
                nb2_top = z_inter*x2[i_layer+1]*Omega_b*emDG0
                n3b1_top = z_inter*(x2[i_layer]*(x1[i_layer+1]+x2[i_layer+1])+x1[i_layer+1]*x2[i_layer+1])*Omega_b*Omega_l*emDG0*emDGT
                n3b2_top = z_inter*(x1[i_layer]*(x1[i_layer+1]+x2[i_layer+1])+x1[i_layer+1]*x2[i_layer+1])*Omega_b*Omega_l*emDG0*emDGT
                n3tot = x1[i_layer]*x2[i_layer]*(x1[i_layer+1]+x2[i_layer+1])+x1[i_layer+1]*x2[i_layer+1]*(x1[i_layer]+x2[i_layer])
                n3tot *= z_inter*Omega_b*Omega_l*emDG0*emDGT
            else:
                if(np.mod(i_layer,2)==0):
                    nb1_top = z_inter*x1[i_layer+1]*Omega_b*emDG0
                    nb2_top = z_inter*x2[i_layer+1]*Omega_b*emDG0
                    n3b1_top = z_inter*(x2[i_layer]*x2[i_layer+1]+x1[i_layer+1]*x2[i_layer+1])*Omega_b*Omega_l*emDG0*emDGT
                    n3b2_top = z_inter*x1[i_layer]*x2[i_layer+1]*Omega_b*Omega_l*emDG0*emDGT
                    n3tot = x1[i_layer]*x2[i_layer]*x2[i_layer+1]+x1[i_layer+1]*x2[i_layer+1]*x1[i_layer]
                    n3tot *= z_inter*Omega_b*Omega_l*emDG0*emDGT
                else:
                    nb1_top = z_inter*x1[i_layer+1]*Omega_b*emDG0
                    nb2_top = z_inter*x2[i_layer+1]*Omega_b*emDG0
                    n3b2_top = z_inter*(x1[i_layer]*x1[i_layer+1]+x1[i_layer+1]*x2[i_layer+1])*Omega_b*Omega_l*emDG0*emDGT
                    n3b1_top = z_inter*x2[i_layer]*x1[i_layer+1]*Omega_b*Omega_l*emDG0*emDGT
                    n3tot = x1[i_layer]*x2[i_layer]*x1[i_layer+1]+x1[i_layer+1]*x2[i_layer+1]*x2[i_layer]
                    n3tot *= z_inter*Omega_b*Omega_l*emDG0*emDGT
                    
    return nb1_top, nb2_top, n3b1_top, n3b2_top, n3tot

def latb_loop(equilib,i_layer,x1,x2,Omega_b,Omega_bs,Omega_l,emDG0,emDGT,z_intra):
    
    nb1_lat=z_intra*x2[i_layer]*Omega_b*emDG0
    nb2_lat=z_intra*x1[i_layer]*Omega_b*emDG0
    if(i_layer == 0 or equilib == False):
        nb1_lat = 0.
        nb2_lat = 0.
        
    n1_loop=x2[i_layer]*Omega_l*emDG0
    n2_loop=x1[i_layer]*Omega_l*emDG0
    if(i_layer == 0):
        n1_loop = 0.
        n2_loop = 0.
        
    return nb1_lat, nb2_lat, n1_loop, n2_loop
    

In [ ]:

# run ###################################

NLmax=12

NL=2

xbulk=10.
xbulk_ref=10.
xbulk, xbulk_ref, fbulk_e, fbulk_e_ref, rhotrans = transition_dens(xbulk,xbulk_ref,NDNA,Omega_b,Omega_l,emDG0,np.exp(-3.),z)

free_en_layer=np.array([])
ncoll=np.array([])

equilib=False

while(NL<NLmax):

    x1=np.array([])
    x2=np.array([])
    x1_old=np.array([])
    x2_old=np.array([])

    for i_layer in np.arange(NL):
        x1=np.append(x1,10.) 
        x2=np.append(x2,10.)
        x1_old=np.append(x1,10.) 
        x2_old=np.append(x2,10.)

# bottom plane  
    x1[0]=Nrec

# fixed point iteration 

    cut=0.0000000000001
    rel_error_vec=np.ones((NL,2))
    n_iter=0

    while(np.max(rel_error_vec) > cut):

        n_iter+=1
        if(np.mod(n_iter,100000) == 0 ): 
            print(n_iter,np.max(rel_error_vec))

        for i_layer in np.arange(NL):
        
            ## the actual number of complexes are nbi_XYY xi[i_layer] ##
        
            # bottom bridges
        
            nb1_bottom, nb2_bottom, n3b1_bottom, n3b2_bottom, n3tot = bottom_bridges(equilib,i_layer,x1,x2,Omega_b,Omega_bs,Omega_l,emDG0,emDGT,z_inter)
      
            # top bridges 
        
            nb1_top, nb2_top, n3b1_top, n3b2_top, n3tot = top_bridges(equilib,i_layer,x1,x2,Omega_b,Omega_bs,Omega_l,emDG0,emDGT,z_inter)
               
            # lateral bridges and loops
        
            nb1_lat, nb2_lat, n1_loop, n2_loop = latb_loop(equilib,i_layer,x1,x2,Omega_b,Omega_bs,Omega_l,emDG0,emDGT,z_intra)

            # iteration
            NN1=NDNA
            NN2=NDNA
            if(i_layer == 0):
                NN1=Nrec
                NN2=0
            x1[i_layer] = NN1/(1+nb1_bottom+nb1_top+nb1_lat+n1_loop+n3b1_bottom+n3b1_top)
            x2[i_layer] = NN2/(1+nb2_bottom+nb2_top+nb2_lat+n2_loop+n3b2_bottom+n3b2_top)

            # update of the error matrix
            rel_error_vec[i_layer,0]=np.abs(x1[i_layer]-x1_old[i_layer])/np.abs(x1_old[i_layer])
            if(i_layer != 0):
                rel_error_vec[i_layer,1]=np.abs(x2[i_layer]-x2_old[i_layer])/np.abs(x2_old[i_layer])
            else:
                rel_error_vec[i_layer,1]=0.

            x1_old[i_layer]=x1[i_layer]
            x2_old[i_layer]=x2[i_layer]

    print('layer=', NL, 'number of iterations=', n_iter)

    #free energy calculation  

    free_A=0
    free_B=0
    free_C=0
    for i_layer in np.arange(NL):
        
        nb1_bottom, nb2_bottom, n3b1_bottom, n3b2_bottom, n3tot_bottom = bottom_bridges(equilib,i_layer,x1,x2,Omega_b,Omega_bs,Omega_l,emDG0,emDGT,z_inter) 
        nb1_top, nb2_top, n3b1_top, n3b2_top, n3tot_top = top_bridges(equilib,i_layer,x1,x2,Omega_b,Omega_bs,Omega_l,emDG0,emDGT,z_inter)        
        nb1_lat, nb2_lat, n1_loop, n2_loop = latb_loop(equilib,i_layer,x1,x2,Omega_b,Omega_bs,Omega_l,emDG0,emDGT,z_intra)

        if(np.abs(n1_loop*x1[i_layer]-n2_loop*x2[i_layer])/(n1_loop*x1[i_layer]+n2_loop*x2[i_layer])>np.power(10.,-4)):
            print('inconsistency in the calculation of the number of loops')
        
        # free_A/free_B/free_C equivalent ways of calculating the free energy
        
        if(i_layer == 0):
            free_A=free_A+Nrec*np.log(x1[i_layer]/Nrec)
            free_B=free_B+Nrec*np.log(x1[i_layer]/Nrec)
            free_C=free_C+Nrec*np.log(x1[i_layer]/Nrec)
        else:
            free_A=free_A+NDNA*np.log(x1[i_layer]/NDNA)+NDNA*np.log(x2[i_layer]/NDNA)
            free_B=free_B+NDNA*np.log(x1[i_layer]/NDNA)+NDNA*np.log(x2[i_layer]/NDNA)
            free_C=free_C+NDNA*np.log(x1[i_layer]/NDNA)+NDNA*np.log(x2[i_layer]/NDNA)
            
        free_A=free_A+nb1_bottom/2*x1[i_layer]+nb2_bottom/2*x2[i_layer]+n3tot_bottom
        free_A=free_A+nb1_top/2*x1[i_layer]+nb2_top/2*x2[i_layer]+n3tot_top
        free_A=free_A+(nb1_lat/2+n1_loop)*x1[i_layer]+nb2_lat/2*x2[i_layer]

        free_B=free_B+nb1_bottom*x1[i_layer]+nb2_bottom*x2[i_layer]+2*n3tot_bottom
        free_B=free_B+(nb1_lat/2+n1_loop)*x1[i_layer]+nb2_lat/2*x2[i_layer]
        
        free_C=free_C+nb1_top*x1[i_layer]+nb2_top*x2[i_layer]+2*n3tot_top
        free_C=free_C+(nb1_lat/2+n1_loop)*x1[i_layer]+nb2_lat/2*x2[i_layer]

    # print(free_A-(NL-1)*fbulk_e_ref,free_B-(NL-1)*fbulk_e_ref,free_C, fbulk_e_ref)
    free_en_layer=np.append(free_en_layer,free_A-(NL-1)*fbulk_e_ref)
    ncoll=np.append(ncoll,NL-1)
    print(' ')
    NL=NL+1

## debug => change indentation when using this block  
#print('check: total number of strands')
#print(' ')

#for i_layer in np.arange(NL):
    
#    print('')
#    print('layer=',i_layer)
    
#    nb1_bottom, nb2_bottom, n3b1_bottom, n3b2_bottom, n3tot = bottom_bridges(equilib,i_layer,x1,x2,Omega_b,Omega_bs,Omega_l,emDG0,emDGT,z_inter)
    
#    print('nb1_bottom=', nb1_bottom*x1[i_layer], 'nb2_bottom=' , nb2_bottom*x2[i_layer], 'n3b1_bottom=', n3b1_bottom*x1[i_layer], 'n3b2_bottom=', n3b2_bottom*x2[i_layer], n3tot)

#    nb1_top, nb2_top, n3b1_top, n3b2_top, n3tot = top_bridges(equilib,i_layer,x1,x2,Omega_b,Omega_bs,Omega_l,emDG0,emDGT,z_inter)

#    print('nb1_top=', nb1_top*x1[i_layer], 'nb2_top=' , nb2_top*x2[i_layer], 'n3b1_top=', n3b1_top*x1[i_layer], 'n3b2_top=', n3b2_top*x2[i_layer], n3tot)
    
#    nb1_lat, nb2_lat, n1_loop, n2_loop = latb_loop(equilib,i_layer,x1,x2,Omega_b,Omega_bs,Omega_l,emDG0,emDGT,z_inter)
    
#    print('nb1_lat=', nb1_lat*x1[i_layer], 'nb2_lat=', nb2_lat*x2[i_layer])
        
#    n1tot=x1[i_layer]*(1.+nb1_bottom+nb1_top+nb1_lat+n1_loop+n3b1_top+n3b1_bottom)
#    n2tot=x2[i_layer]*(1.+nb2_bottom+nb2_top+nb2_lat+n2_loop+n3b2_top+n3b2_bottom)
    
#    print('ilayer=',i_layer,'n1=',n1tot,'n2=',n2tot)
#    print(' ')
        

In [ ]:
print(rhotrans)

In [ ]:

rho=5*np.power(10.,-8)
prob_layer=np.exp(-free_en_layer)*np.power(rho,ncoll)

plt.plot(ncoll,prob_layer)
plt.title('p_layer')
plt.legend()
plt.show()

plt.plot(ncoll,free_en_layer/ncoll)
plt.title('p_layer')
plt.legend()
plt.show()

In [ ]:

rho=5*np.power(10.,-8)
prob_layer=np.exp(-free_en_layer)*np.power(rho,ncoll)

plt.plot(ncoll,prob_layer)
plt.title('p_layer')
plt.legend()
plt.show()

plt.plot(ncoll,free_en_layer/ncoll)
plt.title('p_layer')
plt.legend()
plt.show()


In [ ]:
print(x1)
print(x2)

In [ ]:

# single-strand complexes ##############################################################################

x1_layer = np.array([])
x2_layer = np.array([])
list_layer=np.array([])

for i_layer in np.arange(NL):
    if(i_layer>0):
        x1_layer = np.append(x1_layer,x1[i_layer])
        x2_layer = np.append(x2_layer,x2[i_layer])
        list_layer=np.append(list_layer,i_layer)

plt.plot(list_layer,x1_layer,label='x1, x=a, b')
plt.plot(list_layer,x2_layer,label='x2, x=a, b')
plt.title('single-strand complexes')
plt.legend()
plt.show()

# two- and three-strand complexes ########################################################################

nb_inter_layer=np.array([])
nb_intra_layer=np.array([]) # number of 2-bridges between 'i_layer' and 'i_layer-1'
n_loop=np.array([])
n3b_inter_layer=np.array([]) # number of 3-bridges between 'i_layer' and 'i_layer-1'
list_layer=np.array([])

for i_layer in np.arange(NL):
    if(i_layer>0):
        
        print('layer ',i_layer, ' ----------')
        nb1_bottom, nb2_bottom, n3b1_bottom, n3b2_bottom, n3tot_bottom = bottom_bridges(equilib,i_layer,x1,x2,Omega_b,Omega_bs,Omega_l,emDG0,emDGT,z_inter)
        nb1_top, nb2_top, n3b1_top, n3b2_top, n3tot_top = top_bridges(equilib,i_layer-1,x1,x2,Omega_b,Omega_bs,Omega_l,emDG0,emDGT,z_inter)
        nb1_lat, nb2_lat, n1_loop, n2_loop = latb_loop(equilib,i_layer,x1,x2,Omega_b,Omega_bs,Omega_l,emDG0,emDGT,z_inter)
        
        # inter-plane bridges 
        
        print('1-check inter-plane bidges',nb1_top*x1[i_layer-1],nb1_bottom*x1[i_layer])
        print('2-check inter-plane bidges',nb2_top*x2[i_layer-1],nb2_bottom*x2[i_layer])
        
        nb_bottom=nb1_bottom*x1[i_layer]+nb2_bottom*x2[i_layer]
        nb_inter_layer=np.append(nb_inter_layer,nb_bottom)

        #intra-plane bridges
        
        nb_lateral=x1[i_layer]*nb1_lat+x2[i_layer]*nb2_lat
        nb_intra_layer=np.append(nb_intra_layer,nb_lateral)
        
        # loops
        
        print('check loop',n1_loop*x1[i_layer],n2_loop*x2[i_layer])
        n_l=n1_loop*x1[i_layer]
        n_loop=np.append(n_loop,n_l)
        
        # inter-plane 3-bridges
    
        
        print('check inter-plane 3-bidges',n3tot_bottom,n3tot_top)
        
        n3b_inter_layer=np.append(n3b_inter_layer,n3tot_bottom)
        
        list_layer=np.append(list_layer,i_layer)

plt.plot(list_layer,nb_inter_layer,label='inter-layer')
plt.plot(list_layer,nb_intra_layer,label='intra-layer')
plt.plot(list_layer,n_loop,label='loops')
plt.title('two-strand complexes')
plt.legend()
plt.show()

# three-strand complexes

plt.plot(list_layer,n3b_inter_layer,label='inter-layer')
plt.title('three-strand complexes')
plt.legend()
plt.show()


In [ ]:
print(n3b_inter_layer)